# positive_control_train

Memorization positive control: LoRA rank 8 at learning rate 1e-04 on the 302 rows of
`screen.jsonl`, trained and scored on the same rows.

The control tests whether the training-to-metric chain can move the metric upward at all. Every
arm in the thesis lands at or below the unadapted baseline, which leaves "these methods do not
help here" and "this setup would not show an improvement even if one existed" indistinguishable.
Generalization is not under test; memorization is the point, so training and evaluation share
the same rows.

Rate from `positive_control_lr_probe`, selected on adapter movement rather than training loss,
which is uninformative at a 20-step horizon. At 1e-04 the adapter reaches in 20 steps the
`lora_B` norm the shipped 4,000-step schedule reaches at its end.

Deviations from `options/ft_lora.yml`, all printed in section 5: learning rate 1e-04 against the
shipped 8e-07; 500 steps against 4,000; `warmup_t` 50 against 500, since at 500 total steps the
shipped value would leave the entire run inside warmup; checkpoints every 100 steps; the three
resolution keys at 512; `gradient_accumulation_steps: 8`, required by train.py's own assertion on
a single GPU; and tensorboard-only logging.

No validation-loss module: there is no held-out split here, and validation loss does not track the
GPT-4.1 task metric on this pipeline.

A100 80GB, peak VRAM 45.4 GB. 500 steps at the measured 15.41 s/step is roughly 2.1 hours.
Checkpoints and logs are written to Drive, so a dropped session resumes from the last checkpoint.

## 1. Repo and dependencies

In [ ]:
import os, subprocess, sys, json, shutil

REPO = "/content/OmniGen2"
COMMIT = "18e6f9d5271b517fcb32e999f10df943ae9b8f20"

if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/VectorSpaceLab/OmniGen2.git", REPO], check=True)
    print("cloned")
else:
    print("repo already present")

subprocess.run(["git", "-C", REPO, "checkout", "-q", COMMIT], check=True)
print("commit:", subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"],
                                capture_output=True, text=True).stdout.strip())

# requirements.txt omits torchdiffeq, imported by omnigen2/transport/integrators.py
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=REPO, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchdiffeq"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "huggingface_hub[hf_xet]>=0.30.0,<1.0"], check=True)

# peft raises rather than returning False when torchao predates 0.16.0; Colab ships 0.10.0.
# torchao is unused by the bf16 LoRA path.
_ao = subprocess.run([sys.executable, "-c",
                      "from peft.import_utils import is_torchao_available; is_torchao_available()"],
                     capture_output=True, text=True)
if _ao.returncode != 0:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
    print("torchao removed")

_probe = ("from peft import LoraConfig;"
          "from peft.import_utils import is_torchao_available; is_torchao_available();"
          "from omnigen2.transport import create_transport;"
          "from omnigen2.dataset.omnigen2_train_dataset import OmniGen2TrainDataset;"
          "print('train.py imports OK')")
chk = subprocess.run([sys.executable, "-c", _probe], cwd=REPO, capture_output=True, text=True)
print(chk.stdout.strip() or chk.stderr.strip()[-1000:])
assert chk.returncode == 0, "train.py imports are broken"

sys.path.insert(0, REPO)
os.chdir(REPO)

import torch
print("gpu :", torch.cuda.get_device_name(0),
      "|", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
assert torch.cuda.get_device_properties(0).total_memory / 1e9 > 60, \
    "peak usage is 45.4 GB; an 80 GB A100 is required"

cloned
commit: 18e6f9d5271b517fcb32e999f10df943ae9b8f20
torchao removed
train.py imports OK
gpu : NVIDIA A100-SXM4-80GB | 85.1 GB


## 2. Drive and images

Extracted straight from Drive rather than copied locally first: holding a 48 GB archive and
48 GB of extracted files at once overflows the Colab disk when model weights are also cached.

In [ ]:
from google.colab import drive, userdata

drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
IMGS, EXPECTED = "/content/imgs", 37063

def n_files(d):
    return sum(len(f) for _, _, f in os.walk(d)) if os.path.isdir(d) else 0

if n_files(IMGS) < EXPECTED:
    free_gb = shutil.disk_usage("/content").free / 1e9
    print(f"free before restore: {free_gb:.0f} GB")
    assert free_gb > 70, f"only {free_gb:.0f} GB free; the image set needs ~50 GB plus model cache"
    os.makedirs(IMGS, exist_ok=True)
    subprocess.run(f'tar -xf "{DRIVE}/imgs_subset.tar" -C {IMGS}', shell=True, check=True)
    assert n_files(IMGS) >= EXPECTED, f"restore incomplete: {n_files(IMGS)} of {EXPECTED}"
print("images:", n_files(IMGS))

for f in ["screen.jsonl", "test.jsonl", "val.jsonl"]:
    if not os.path.exists(f"/content/{f}"):
        subprocess.run(["cp", f"{DRIVE}/{f}", "/content/"], check=True)

print(subprocess.run(["df", "-h", "/content"], capture_output=True, text=True).stdout.strip().split("\n")[-1])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
free before restore: 193 GB
images: 37063
overlay         236G  101G  135G  43% /


## 3. Memorization set

Written by `positive_control_lr_probe` and reused unchanged. Rebuilt here only if absent, with
the same leakage assertions against the test and validation targets.

In [ ]:
import yaml
from omegaconf import OmegaConf

CTRL = f"{DRIVE}/lora"
os.makedirs(f"{CTRL}/jsonls", exist_ok=True)
MEM_JSONL = f"{CTRL}/jsonls/memorize_302.jsonl"

screen = [json.loads(l) for l in open("/content/screen.jsonl", encoding="utf-8")]
test_t = {json.loads(l)["output_image"] for l in open("/content/test.jsonl", encoding="utf-8")}
val_t  = {json.loads(l)["output_image"] for l in open("/content/val.jsonl",  encoding="utf-8")}
scr_t  = {r["output_image"] for r in screen}

assert len(screen) == 302, f"expected 302 screen rows, got {len(screen)}"
assert len(scr_t) == 62, f"expected 62 targets, got {len(scr_t)}"
assert not (scr_t & test_t), "screen leaks into test"
assert not (scr_t & val_t), "screen leaks into val"

def convert(r):
    return {"task_type": "ic",
            "instruction": r["instruction"],
            "input_images": [os.path.join(IMGS, p) for p in r["input_images"]],
            "output_image": os.path.join(IMGS, r["output_image"])}

if not os.path.exists(MEM_JSONL):
    with open(MEM_JSONL, "w", encoding="utf-8") as f:
        for r in screen:
            f.write(json.dumps(convert(r), ensure_ascii=False) + "\n")
    print("wrote", os.path.basename(MEM_JSONL), len(screen), "rows")
else:
    print("jsonl already present")

# The Drive path contains an apostrophe, which closes a single-quoted YAML scalar early.
# json.dumps emits a correctly escaped double-quoted scalar, which YAML accepts verbatim.
def _cfg(target):
    return 'data:\n  - \n    path: %s\n    type: "ic"\n    ratio: !!float 1\n' % json.dumps(target)

def _write_if_changed(path, body):
    try:
        if open(path, encoding="utf-8").read() == body:
            return False
    except FileNotFoundError:
        pass
    open(path, "w", encoding="utf-8").write(body)
    return True

for _name, _target in [("memorize_ic.yml",  MEM_JSONL),
                       ("memorize_mix.yml", f"{CTRL}/memorize_ic.yml")]:
    _p = f"{CTRL}/{_name}"
    _changed = _write_if_changed(_p, _cfg(_target))
    _parsed = yaml.safe_load(open(_p, encoding="utf-8"))["data"][0]["path"]
    assert _parsed == _target, f"{_name} parsed to {_parsed}"
    print(f"{_name}: {'written' if _changed else 'unchanged'}, parses to {os.path.basename(_target)}")

rows = [json.loads(l) for l in open(MEM_JSONL, encoding="utf-8")]
missing = [p for r in rows for p in r["input_images"] + [r["output_image"]] if not os.path.exists(p)]
assert not missing, f"{len(missing)} missing images, first: {missing[:2]}"
print(f"{len(rows)} rows, all images present, {len(rows)/16:.1f} steps per epoch at global batch 16")

jsonl already present
memorize_ic.yml: unchanged, parses to memorize_302.jsonl
memorize_mix.yml: unchanged, parses to memorize_ic.yml
302 rows, all images present, 18.9 steps per epoch at global batch 16


## 4. Patch train.py

Two edits, both carried over from the probe. `FIX_ACCUM_NORM` matches the gradient scale every
other arm in the thesis ran at. The second prints the LoRA `B` norm every ten steps, which is
zero at init, so the console log records the adapter's growth curve without a conversion step.

Idempotent. `git checkout -- train.py` reverts, in section 7.

In [ ]:
p = f"{REPO}/train.py"
src = open(p, encoding="utf-8").read()

# The "* accelerator.gradient_state.num_steps" factor in the loss normalisation is a no-op on
# the reference 8-GPU setup, where num_steps is 1; with accumulation 8 on one GPU it becomes an
# active multiplier and inflates the gradient by exactly the accumulation factor. True
# reproduces the reference gradient scale, as in every other arm.
FIX_ACCUM_NORM = True

A3     = "                    accelerator.clip_grad_norm_(trainable_params, args.train.max_grad_norm)\n"
A3_NEW = "                    _gnorm = accelerator.clip_grad_norm_(trainable_params, args.train.max_grad_norm)\n"
A4     = "                loss = (loss * accelerator.gradient_state.num_steps * accelerator.num_processes) / num_tokens_in_batch\n"
A4_NEW = "                loss = (loss * accelerator.num_processes) / num_tokens_in_batch\n"
B1     = "                global_step += 1\n"
B1_NEW = B1 + '''                if accelerator.is_main_process and global_step % 10 == 0:
                    try:
                        _bn = sum(float((_bp.detach().float() ** 2).sum())
                                  for _bk, _bp in accelerator.unwrap_model(model).named_parameters()
                                  if "lora_B" in _bk) ** 0.5
                    except Exception:
                        _bn = float("nan")
                    print(f"[ctrl] step {global_step} b_norm {_bn:.6e} "
                          f"grad_norm {float(_gnorm):.4e}", flush=True)
'''

if "[ctrl] step" not in src:
    assert src.count(A3) == 1, "clip anchor moved"
    assert src.count(A4) == 1, "loss-normalisation anchor moved"
    assert src.count(B1) == 1, "global_step anchor moved"
    src = src.replace(A3, A3_NEW).replace(B1, B1_NEW)
    if FIX_ACCUM_NORM:
        src = src.replace(A4, A4_NEW)
    open(p, "w", encoding="utf-8").write(src)

now = open(p, encoding="utf-8").read()
print("  pre-clip norm capture :", A3_NEW.strip() in now)
print("  b_norm trace          :", "[ctrl] step" in now)
print("  accum-norm fix        :", A4_NEW.strip() in now, f"(FIX_ACCUM_NORM={FIX_ACCUM_NORM})")

  pre-clip norm capture : True
  b_norm trace          : True
  accum-norm fix        : True (FIX_ACCUM_NORM=True)


## 5. Configuration

Every deviation from `options/ft_lora.yml` is printed with its reason. Unlisted keys keep their
shipped values, including `global_batch_size: 16`, `batch_size: 2`, `max_grad_norm: 1`,
`lora_dropout`, `gradient_checkpointing`, `ref_img_dropout_prob: 0.5`, `prompt_dropout_prob` and
`seed: 2233`.

`ref_img_dropout_prob: 0.5` does not drop half the references. The dataset computes
`drop_ref_img = drop_prompt and random.random() < ref_img_dropout_prob`, so it is gated on
prompt dropout at 1e-4 and the effective rate is 5e-5: under one row in this entire run.

In [ ]:
LORA_RANK = 8
LR        = 1e-4
STEPS     = 500
WARMUP    = 50
CKPT_EVERY = 100

RUN = "positive_control_r8"
EXP = f"{CTRL}/experiments"
os.makedirs(EXP, exist_ok=True)

# train.py writes to <repo>/experiments/<name>; symlink that to Drive so a dropped
# session resumes from the last checkpoint
link = f"{REPO}/experiments"
if os.path.islink(link) or os.path.exists(link):
    if not (os.path.islink(link) and os.readlink(link) == EXP):
        os.unlink(link) if os.path.islink(link) else shutil.rmtree(link)
        os.symlink(EXP, link)
else:
    os.symlink(EXP, link)
print("experiments ->", os.readlink(link))

base = OmegaConf.load(f"{REPO}/options/ft_lora.yml")
c    = OmegaConf.load(f"{REPO}/options/ft_lora.yml")

c.name = RUN
c.data.data_path = f"{CTRL}/memorize_mix.yml"
c.data.max_output_pixels = 262144          # 512 * 512
c.data.max_input_pixels = [262144] * 4
c.data.max_side_length = 512
c.train.gradient_accumulation_steps = 8
c.train.lora_rank = LORA_RANK
c.train.learning_rate = LR
c.train.warmup_t = WARMUP
c.train.max_train_steps = STEPS
c.logger.log_with = ["tensorboard"]
c.logger.checkpointing_steps = CKPT_EVERY

CFG = f"{REPO}/options/{RUN}.yml"
OmegaConf.save(c, CFG)

reasons = {
    "name": "per-run experiment directory",
    "data.data_path": "the 302 memorization rows, trained and scored on the same rows",
    "data.max_output_pixels": "512x512 output, matching every generated image in the thesis",
    "data.max_input_pixels": "references at max side 512, matching load_inputs at inference",
    "data.max_side_length": "512",
    "train.gradient_accumulation_steps": "required: batch 2 x accum 8 x 1 GPU = global 16, asserted by train.py",
    "train.lora_rank": "the rank the headline arm reports",
    "train.learning_rate": "selected by positive_control_lr_probe on adapter movement",
    "train.warmup_t": "at 500 total steps the shipped 500 would leave the whole run inside warmup",
    "train.max_train_steps": "26 epochs over 302 rows",
    "logger.checkpointing_steps": "5 checkpoints, since the adapter moves fast at this rate",
    "logger.log_with": "wandb would block on an interactive login",
}

print(f"{'key':38s} {'shipped':<12s} -> {'this run':<12s} reason")
print("-" * 132)
for k, why in reasons.items():
    print(f"{k:38s} {str(OmegaConf.select(base, k)):<12s} -> {str(OmegaConf.select(c, k)):<12s} {why}")
print(f"{'val_loss.*':38s} {'absent':<12s} -> {'absent':<12s} "
      f"no held-out split here, and val loss does not track the task metric on this pipeline")

print("\nunchanged:", ", ".join(
    f"{k}={OmegaConf.select(c, k)}" for k in
    ["train.global_batch_size", "train.batch_size", "train.lora_dropout",
     "train.gradient_checkpointing", "train.max_grad_norm",
     "data.ref_img_dropout_prob", "data.prompt_dropout_prob", "data.maximum_text_tokens", "seed"]))
print(f"\nconfig: {CFG}")
print(f"{STEPS} steps at the measured 15.41 s/step: {STEPS*15.41/3600:.1f} h "
      f"= {STEPS*15.41/3600*11.77:.0f} units")

experiments -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/lora/experiments
key                                    shipped      -> this run     reason
------------------------------------------------------------------------------------------------------------------------------------
name                                   ft_lora      -> positive_control_r8 per-run experiment directory
data.data_path                         data_configs/train/example/mix.yml -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/lora/memorize_mix.yml the 302 memorization rows, trained and scored on the same rows
data.max_output_pixels                 1048576      -> 262144       512x512 output, matching every generated image in the thesis
data.max_input_pixels                  [1048576, 1048576, 589824, 262144] -> [262144, 262144, 262144, 262144] references at max side 512, matching load_inputs at inference
data.max_side_length                   2048         -> 512         

## 6. Train

`resume_from_checkpoint: latest` is a shipped default, so re-running this cell after a dropped
session continues from the last checkpoint rather than restarting.

In [ ]:
import time, re

LAUNCH = ["accelerate", "launch",
          "--num_processes=1", "--num_machines=1", "--machine_rank=0", "--main_process_port=29500",
          "--use_fsdp", "--fsdp_offload_params", "false",
          "--fsdp_sharding_strategy", "HYBRID_SHARD_ZERO2",
          "--fsdp_auto_wrap_policy", "TRANSFORMER_BASED_WRAP",
          "--fsdp_transformer_layer_cls_to_wrap", "OmniGen2TransformerBlock",
          "--fsdp_state_dict_type", "FULL_STATE_DICT",
          "--fsdp_forward_prefetch", "false",
          "--fsdp_use_orig_params", "True",
          "--fsdp_cpu_ram_efficient_loading", "false",
          "--fsdp_sync_module_states", "True"]

os.makedirs(f"{EXP}/{RUN}", exist_ok=True)
LOG = f"{EXP}/{RUN}/train_console.log"

t0 = time.time()
proc = subprocess.Popen(LAUNCH + ["train.py", "--config", CFG], cwd=REPO,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, env={**os.environ, "WANDB_DISABLED": "true"})

with open(LOG, "a", encoding="utf-8") as lf:
    for line in proc.stdout:
        lf.write(line)
        if re.search(r"\[ctrl\]|Saving state|steps_per_second|Traceback|Error|error|"
                     r"Number of training samples", line):
            print(f"[{(time.time()-t0)/3600:5.2f}h] {line.rstrip()}", flush=True)
proc.wait()

print(f"\nexit {proc.returncode} after {(time.time()-t0)/3600:.2f}h")
if proc.returncode != 0:
    print("\n--- tail ---")
    print("".join(open(LOG, encoding="utf-8").readlines()[-40:]))

[ 0.00h] 2026-09-11 12:30:37.729008: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
[ 0.03h] 09/11/2026 12:32:22 - INFO - __main__ - [RANK 0] Number of training samples: 302
[ 0.08h] [ctrl] step 10 b_norm 5.845200e-02 grad_norm 1.7886e-02
[ 0.12h] [ctrl] step 20 b_norm 2.208926e-01 grad_norm 1.9725e-02
[ 0.16h] [ctrl] step 30 b_norm 4.728863e-01 grad_norm 2.6378e-02
[ 0.21h] [ctrl] step 40 b_norm 7.691545e-01 grad_norm 2.3733e-02
[ 0.25h] [ctrl] step 50 b_norm 1.078580e+00 grad_norm 1.4422e-02
[ 0.29h] [ctrl] step 60 b_norm 1.351970e+00 grad_norm 1.5908e-02
[ 0.33h] [ctrl] step 70 b_norm 1.570544e+00 grad_norm 1.6501e-02
[ 0.37h] [ctrl] step 80 b_norm 1.748390e+00 grad_norm 1.9378e-02
[ 0.41h] [ctrl] step 90 b_norm 1.910092e+00 grad_norm 3.3518e-02
[ 0.45h] [ctrl] step 1

## 7. Adapter growth against the trained arm

The control has no validation curve, so the diagnostic is the adapter itself. `b_norm` from the
console log is compared against the `lora_B` norms of `lora_r8`, read from its converted
checkpoints on Drive, which puts this run's movement on the scale of the arm it underwrites.

A control that has not moved past the arm's own trajectory has not been trained hard enough for
memorization to be the expected outcome.

In [ ]:
import glob
from safetensors.torch import load_file

LOGTXT = open(LOG, encoding="utf-8").read() if os.path.exists(LOG) else ""
trace = [(int(s), float(b)) for s, b in re.findall(r"\[ctrl\] step (\d+) b_norm (\S+)", LOGTXT)]
gnorm = [float(g) for g in re.findall(r"\[ctrl\] step \d+ b_norm \S+ grad_norm (\S+)", LOGTXT)]

print("=== 1. reference: lora_B norms of the trained arm ===")
ref = {}
for d in sorted(glob.glob(f"{EXP}/lora_r8*/checkpoint-*/transformer_lora/pytorch_lora_weights.safetensors")):
    run_name = d.split("/experiments/")[1].split("/")[0]
    step = int(d.split("checkpoint-")[1].split("/")[0])
    t = load_file(d)
    n = float(torch.cat([v.float().flatten() for k, v in sorted(t.items()) if "lora_B" in k]).norm())
    ref[(run_name, step)] = n
    print(f"  {run_name:16s} step {step:>4d}  ||B|| = {n:.4f}")
if not ref:
    print("  no converted lora_r8 checkpoints found on Drive; comparison skipped")

print("\n=== 2. this run ===")
for s, b in trace:
    over = ""
    if ref:
        best = max(v for v in ref.values())
        over = f"  ({100*b/best:.0f}% of the arm's step-4000 norm)"
    print(f"  step {s:>4d}  ||B|| = {b:.4f}{over}")
print(f"  -> {'OK' if trace else 'NO TRACE - check that the section 4 patch landed'}")

print("\n=== 3. gradient norm ===")
if gnorm:
    import statistics as st
    print(f"  pre-clip norms: {[round(x, 4) for x in gnorm[:8]]}")
    print(f"  median {st.median(gnorm):.4f} against max_grad_norm={c.train.max_grad_norm}")
    print(f"  clipping active: {max(gnorm) > c.train.max_grad_norm}")
else:
    print("  none logged")

print("\n=== 4. checkpoints on Drive ===")
d0 = f"{EXP}/{RUN}"
ck = sorted((d for d in os.listdir(d0) if d.startswith("checkpoint-")),
            key=lambda x: int(x.split("-")[1])) if os.path.isdir(d0) else []
for d in ck:
    sz = sum(os.path.getsize(os.path.join(r, f))
             for r, _, fs in os.walk(f"{d0}/{d}") for f in fs) / 1e9
    print(f"  {d}: {sz:.2f} GB")
print(f"  -> {'OK' if len(ck) == STEPS // CKPT_EVERY else str(len(ck)) + ' of ' + str(STEPS // CKPT_EVERY)}")
print("\nnext: positive_control_convert, then generation on the same 302 rows")

=== 1. reference: lora_B norms of the trained arm ===
  lora_r8          step 1000  ||B|| = 0.2688
  lora_r8          step 2000  ||B|| = 0.5504
  lora_r8          step 3000  ||B|| = 0.7574
  lora_r8          step 4000  ||B|| = 0.9188
  lora_r8_early    step 1000  ||B|| = 0.2693
  lora_r8_early    step  250  ||B|| = 0.0243
  lora_r8_early    step  500  ||B|| = 0.0944
  lora_r8_early    step  750  ||B|| = 0.1858

=== 2. this run ===
  step   10  ||B|| = 0.0585  (6% of the arm's step-4000 norm)
  step   20  ||B|| = 0.2209  (24% of the arm's step-4000 norm)
  step   30  ||B|| = 0.4729  (51% of the arm's step-4000 norm)
  step   40  ||B|| = 0.7692  (84% of the arm's step-4000 norm)
  step   50  ||B|| = 1.0786  (117% of the arm's step-4000 norm)
  step   60  ||B|| = 1.3520  (147% of the arm's step-4000 norm)
  step   70  ||B|| = 1.5705  (171% of the arm's step-4000 norm)
  step   80  ||B|| = 1.7484  (190% of the arm's step-4000 norm)
  step   90  ||B|| = 1.9101  (208% of the arm's step-4000 

## 8. Revert the patch

Leaves the repo clean so the next session re-clones or reuses an unmodified `train.py`.

In [ ]:
subprocess.run(["git", "-C", REPO, "checkout", "--", "train.py"], check=True)
src = open(f"{REPO}/train.py", encoding="utf-8").read()
assert "[ctrl] step" not in src and "_gnorm" not in src
assert A4 in src, "loss normalisation not restored"
print("train.py reverted to", COMMIT[:7])

train.py reverted to 18e6f9d
